In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, QED
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

class MolecularFeatureExtractor:
    """Extract molecular features from SMILES strings."""

    def __init__(self, max_features=2048):
        """Initialize feature extractor with Morgan fingerprint settings."""
        self.max_features = max_features

    def smiles_to_fingerprint(self, smiles):
        """Convert SMILES string to Morgan fingerprint."""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=self.max_features)
            return np.array(fingerprint)
        except:
            return None

    def extract_features(self, smiles_list):
        """Extract features for a list of SMILES strings."""
        features = []
        valid_smiles = []

        for smiles in tqdm(smiles_list, desc="Extracting features"):
            fp = self.smiles_to_fingerprint(smiles)
            if fp is not None:
                features.append(fp)
                valid_smiles.append(smiles)

        return np.array(features), valid_smiles

    def calculate_properties(self, smiles):
        """Calculate molecular properties for a SMILES string."""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None

            properties = {
                'MW': Descriptors.MolWt(mol),
                'LogP': Descriptors.MolLogP(mol),
                'HBA': Descriptors.NumHAcceptors(mol),
                'HBD': Descriptors.NumHDonors(mol),
                'TPSA': Descriptors.TPSA(mol),
                'QED': QED.qed(mol),
                'NumRotBonds': Descriptors.NumRotatableBonds(mol)
            }
            return properties
        except:
            return None


class DrugGAN:
    """GAN model for generating novel drug-like molecules."""

    def __init__(self, input_dim=2048, latent_dim=128):
        """Initialize the GAN architecture."""
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Create optimizer
        self.optimizer = Adam(learning_rate=0.0002, beta_1=0.5)

        # Build and compile the discriminator
        self.discriminator = self._build_discriminator()
        self.discriminator.compile(
            loss='binary_crossentropy',
            optimizer=self.optimizer,
            metrics=['accuracy']
        )

        # Build the generator
        self.generator = self._build_generator()

        # The generator takes noise as input and generates fake samples
        z = tf.keras.Input(shape=(self.latent_dim,))
        fake_sample = self.generator(z)

        # For the combined model we will only train the generator
        self.discriminator.trainable = False

        # The discriminator takes generated samples as input and determines validity
        validity = self.discriminator(fake_sample)

        # The combined model (stacked generator and discriminator)
        self.combined = tf.keras.Model(z, validity)
        self.combined.compile(loss='binary_crossentropy', optimizer=self.optimizer)

    def _build_generator(self):
        """Build the generator network."""
        model = Sequential()

        # First dense layer
        model.add(Dense(256, input_dim=self.latent_dim))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        # Second dense layer
        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        # Third dense layer
        model.add(Dense(1024))
        model.add(LeakyReLU(alpha=0.2))
        model.add(BatchNormalization(momentum=0.8))

        # Output layer with sigmoid activation
        model.add(Dense(self.input_dim, activation='sigmoid'))

        model.summary()

        noise = tf.keras.Input(shape=(self.latent_dim,))
        img = model(noise)

        return tf.keras.Model(noise, img)

    def _build_discriminator(self):
        """Build the discriminator network."""
        model = Sequential()

        # First dense layer
        model.add(Dense(1024, input_dim=self.input_dim))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.3))

        # Second dense layer
        model.add(Dense(512))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.3))

        # Third dense layer
        model.add(Dense(256))
        model.add(LeakyReLU(alpha=0.2))
        model.add(Dropout(0.3))

        # Output layer with sigmoid activation
        model.add(Dense(1, activation='sigmoid'))

        model.summary()

        img = tf.keras.Input(shape=(self.input_dim,))
        validity = model(img)

        return tf.keras.Model(img, validity)

    def train(self, X_train, epochs, batch_size=32, save_interval=50):
        """Train the GAN model."""
        # Adversarial ground truths
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))

        d_losses = []
        g_losses = []

        for epoch in range(epochs):
            # ---------------------
            #  Train Discriminator
            # ---------------------

            # Select a random batch of real samples
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            real_samples = X_train[idx]

            # Generate a batch of fake samples
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            fake_samples = self.generator.predict(noise)

            # Train the discriminator
            d_loss_real = self.discriminator.train_on_batch(real_samples, valid)
            d_loss_fake = self.discriminator.train_on_batch(fake_samples, fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # ---------------------
            #  Train Generator
            # ---------------------

            # Generate new noise for generator training
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))

            # Train the generator (to have the discriminator label samples as valid)
            g_loss = self.combined.train_on_batch(noise, valid)

            # Save the losses
            d_losses.append(d_loss[0])
            g_losses.append(g_loss)

            # Print the progress
            print(f"Epoch {epoch+1}/{epochs}, D Loss: {d_loss[0]:.4f}, G Loss: {g_loss:.4f}")

            # If at save interval => save generated samples
            if (epoch + 1) % save_interval == 0:
                self._save_model(epoch + 1)

        return d_losses, g_losses

    def _save_model(self, epoch):
        """Save the model weights at specific epochs."""
        self.generator.save_weights(f"models/generator_epoch_{epoch}.h5")
        self.discriminator.save_weights(f"models/discriminator_epoch_{epoch}.h5")

    def generate_samples(self, num_samples):
        """Generate new molecular fingerprints."""
        noise = np.random.normal(0, 1, (num_samples, self.latent_dim))
        return self.generator.predict(noise)


class MoleculeDecoder:
    """Convert fingerprints back to SMILES strings using a pre-trained model."""

    def __init__(self, model_path):
        """Load the pre-trained decoder model."""
        # In a real implementation, this would load a pre-trained model
        # that converts fingerprints back to SMILES
        # For now, this is a placeholder
        print(f"Loading decoder model from {model_path}")
        self.model = None  # Would load actual model here

    def decode(self, fingerprints, threshold=0.5):
        """Decode fingerprints to SMILES strings."""
        # In a real implementation, this would use the model to generate SMILES
        # For now, we'll just note that this would happen here
        print(f"Decoding {len(fingerprints)} fingerprints to SMILES")
        # Binarize the fingerprints
        binary_fps = (fingerprints > threshold).astype(int)
        # This would return SMILES strings in a real implementation
        return ["C1CCCCC1" for _ in range(len(fingerprints))]  # Placeholder


class DrugDiscoveryPipeline:
    """End-to-end pipeline for drug discovery using GAN."""

    def __init__(self, data_path, decoder_model_path=None):
        """Initialize the pipeline with data source and models."""
        self.data_path = data_path
        self.feature_extractor = MolecularFeatureExtractor()
        self.gan = None
        self.decoder = None if decoder_model_path is None else MoleculeDecoder(decoder_model_path)

    def load_data(self):
        """Load and preprocess chemistry data."""
        print(f"Loading data from {self.data_path}")
        # In a real implementation, this would load real chemistry data
        # For this example, we'll create a synthetic dataset

        try:
            # Try to load from file first
            df = pd.read_csv(self.data_path)
            smiles_list = df['SMILES'].tolist()
        except:
            # If file doesn't exist or has issues, create synthetic data
            print("Creating synthetic data for demonstration")
            # Some example SMILES strings of common drug-like molecules
            smiles_list = [
                "CC(=O)OC1=CC=CC=C1C(=O)O",  # Aspirin
                "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",  # Ibuprofen
                "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",  # Caffeine
                "CC(=O)NC1=CC=C(C=C1)O",  # Acetaminophen/Paracetamol
                "C1=CC=C2C(=C1)C=CC=C2",  # Naphthalene
                "C1=CC=C(C=C1)C(=O)O",  # Benzoic acid
                "COC1=CC=C(C=C1)CCN",  # 4-Methoxyphenethylamine
                "CC1=C(C=CC=C1)C",  # o-Xylene
                "CC1=CC=CC=C1",  # Toluene
                "C1=CC=CC=C1"  # Benzene
            ] * 100  # Replicate to create more data

            # Save the synthetic data
            pd.DataFrame({'SMILES': smiles_list}).to_csv("synthetic_drug_data.csv", index=False)

        return smiles_list

    def prepare_data(self, smiles_list):
        """Extract features and prepare data for training."""
        features, valid_smiles = self.feature_extractor.extract_features(smiles_list)

        # Split data
        X_train, X_test = train_test_split(features, test_size=0.2, random_state=42)

        print(f"Prepared {len(X_train)} training samples and {len(X_test)} test samples")
        return X_train, X_test, valid_smiles

    def train_model(self, X_train, epochs=1000, batch_size=32, save_interval=100):
        """Train the GAN model."""
        # Initialize GAN with appropriate dimensions
        self.gan = DrugGAN(input_dim=X_train.shape[1])

        # Train the model
        d_losses, g_losses = self.gan.train(X_train, epochs, batch_size, save_interval)

        # Plot training losses
        self._plot_training_history(d_losses, g_losses)

        return self.gan

    def _plot_training_history(self, d_losses, g_losses):
        """Plot discriminator and generator losses."""
        plt.figure(figsize=(10, 5))
        plt.plot(d_losses, label='Discriminator Loss')
        plt.plot(g_losses, label='Generator Loss')
        plt.xlabel('Iteration')
        plt.ylabel('Loss')
        plt.legend()
        plt.savefig('gan_training_loss.png')
        plt.close()

    def generate_novel_compounds(self, num_samples=100, threshold=0.5):
        """Generate novel compounds and evaluate their properties."""
        if self.gan is None:
            raise ValueError("GAN model not trained. Call train_model first.")

        # Generate fingerprints
        print(f"Generating {num_samples} novel fingerprints")
        generated_fps = self.gan.generate_samples(num_samples)

        # Decode to SMILES if decoder is available
        if self.decoder is not None:
            generated_smiles = self.decoder.decode(generated_fps, threshold)

            # Calculate properties
            properties_list = []
            valid_smiles = []

            for smiles in generated_smiles:
                props = self.feature_extractor.calculate_properties(smiles)
                if props is not None:
                    properties_list.append(props)
                    valid_smiles.append(smiles)

            # Convert to DataFrame
            if properties_list:
                properties_df = pd.DataFrame(properties_list)
                print(f"Generated {len(valid_smiles)} valid molecules")

                # Plot property distributions
                self._plot_property_distributions(properties_df)

                # Save results
                result_df = pd.DataFrame({'SMILES': valid_smiles})
                for col in properties_df.columns:
                    result_df[col] = properties_df[col]
                result_df.to_csv('generated_molecules.csv', index=False)

                return result_df
            else:
                print("No valid molecules generated")
                return None
        else:
            print("No decoder available. Returning raw fingerprints.")
            return generated_fps

    def _plot_property_distributions(self, properties_df):
        """Plot distributions of molecular properties."""
        plt.figure(figsize=(15, 10))
        for i, col in enumerate(properties_df.columns):
            plt.subplot(2, 4, i+1)
            sns.histplot(properties_df[col], kde=True)
            plt.title(col)
        plt.tight_layout()
        plt.savefig('property_distributions.png')
        plt.close()

    def run_pipeline(self, epochs=1000, batch_size=32, num_samples=100):
        """Run the complete drug discovery pipeline."""
        # Load data
        smiles_list = self.load_data()

        # Prepare data
        X_train, X_test, valid_smiles = self.prepare_data(smiles_list)

        # Train model
        self.train_model(X_train, epochs, batch_size)

        # Generate novel compounds
        generated_molecules = self.generate_novel_compounds(num_samples)

        return generated_molecules


if __name__ == "__main__":
    # Example usage
    data_path = "drug_dataset.csv"  # Path to your chemistry data
    decoder_path = "models/smiles_decoder.h5"  # Path to pre-trained decoder model

    # Initialize pipeline
    pipeline = DrugDiscoveryPipeline(data_path, decoder_path)

    # Run complete pipeline
    results = pipeline.run_pipeline(epochs=200, batch_size=32, num_samples=500)

    if results is not None:
        # Filter for promising drug candidates based on properties
        promising = results[
            (results['MW'] < 500) &  # Lipinski's Rule of Five
            (results['LogP'] < 5) &
            (results['HBD'] <= 5) &
            (results['HBA'] <= 10) &
            (results['QED'] > 0.6)  # Drug-likeness score
        ]

        print(f"Found {len(promising)} promising drug candidates")
        promising.to_csv('promising_candidates.csv', index=False)